# Compréhension Audio — Qwen2-Audio-7B-Instruct

Lancer le backend FastAPI et l'interface web sur Colab (T4, 4-bit NF4), puis y accéder via un tunnel public.

Exécuter les cellules dans l'ordre.

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_URL  = 'https://github.com/mouadelhaddad/Tasnim.git'
CLONE_DIR = '/content/Tasnim'

!rm -rf {CLONE_DIR}
!git clone {REPO_URL} {CLONE_DIR}

BACKEND_DIR  = os.path.join(CLONE_DIR, 'backend')
PROJECT_ROOT = CLONE_DIR
assert os.path.isfile(os.path.join(BACKEND_DIR, 'app', 'main.py'))

with open(os.path.join(BACKEND_DIR, 'app', 'model.py')) as f:
    src = f.read()
if '_build_inputs' not in src:
    print('WARNING: audio fix missing — push the latest code and re-run this cell.')
else:
    print('OK:', BACKEND_DIR)


In [ ]:
!apt-get -qq install -y ffmpeg
!pip install -q "transformers>=4.45.0" accelerate bitsandbytes \
    librosa==0.10.2 soundfile==0.12.1 \
    fastapi==0.115.5 "uvicorn[standard]==0.32.1" python-multipart==0.0.12 \
    "pydantic>=2.0.0" python-dotenv==1.0.1
print('done')


In [ ]:
import os, torch

total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}  ({total_gb:.1f} GB)')

os.environ['MODEL_NAME']   = 'Qwen/Qwen2-Audio-7B-Instruct'
os.environ['USE_MOCK']     = 'false'
os.environ['LOAD_IN_4BIT'] = 'true'
os.environ['LOAD_IN_8BIT'] = 'false'


In [ ]:
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared


In [ ]:
import subprocess, time, re, os

subprocess.run(['pkill', '-9', '-f', 'uvicorn'],     stderr=subprocess.DEVNULL)
subprocess.run(['pkill', '-9', '-f', 'cloudflared'], stderr=subprocess.DEVNULL)
subprocess.run('fuser -k 8000/tcp', shell=True,      stderr=subprocess.DEVNULL)
time.sleep(3)

UVICORN_LOG = '/content/uvicorn.log'
TUNNEL_LOG  = '/content/cloudflared.log'

server = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND_DIR,
    stdout=open(UVICORN_LOG, 'w'), stderr=subprocess.STDOUT,
    env={**os.environ},
)

tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=open(TUNNEL_LOG, 'w'), stderr=subprocess.STDOUT,
)

time.sleep(5)
if server.poll() is not None:
    print('Server crashed:', open(UVICORN_LOG).read()[-1500:])
else:
    public_url = None
    for _ in range(40):
        time.sleep(2)
        try:
            log = open(TUNNEL_LOG).read()
        except FileNotFoundError:
            continue
        m = re.search(r'https://[-\w.]+\.trycloudflare\.com', log)
        if m:
            public_url = m.group(0)
            break
    print(public_url or 'Tunnel URL not found — check /content/cloudflared.log')


In [ ]:
import time, requests

print('Loading', end='', flush=True)
for _ in range(180):
    try:
        r = requests.get('http://localhost:8000/api/v1/health', timeout=5)
        if r.ok and r.json().get('model_loaded'):
            print('\nReady:', r.json())
            print('URL:', public_url)
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(10)
else:
    print('\nTimeout — check the log:')
    print(open('/content/uvicorn.log').read()[-2000:])


In [ ]:
!tail -n 60 /content/uvicorn.log